# 01 — Model Training

Trains 9 models on a given split with two target transforms (raw AOD, log1p AOD).
Set `SPLIT` in the config cell, then run cells top-to-bottom through the model definitions.

**Each transform has its own cell** — run them independently:
- Run **"Train — raw"** cell alone to train/re-train raw models only
- Run **"Train — log1p"** cell alone to train/re-train log1p models only

**Models and feature inputs:**
- GPU boosting (raw ~3,700 features): LightGBM, XGBoost, CatBoost
- CPU trees (raw ~3,700 features): Random Forest, Extra Trees, Gradient Boosting
- Linear/kernel (PCA-150): Ridge, Lasso, SVR-RBF

**TabPFN** is benchmarked separately at the bottom of this notebook.
It tests PCA-150 and MI top-100/200/300/400/500 × both transforms, with fit and predict
timed independently to diagnose performance.

**Outputs:** `results/predictions/{split}/{model}_{transform}.csv` and `models/{split}/{model}_{transform}.joblib`

In [1]:
SPLIT = 'holdout_time'   # 'stratified' | 'holdout_time'

In [2]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import joblib
import sklearn.base
from pathlib import Path
from time import time

from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from tabpfn import TabPFNRegressor

ROOT     = Path('D:/Regional-Scale-AOD-Estimation')
FEAT_DIR = ROOT / 'feature_engineering/features' / SPLIT
PRED_DIR = ROOT / 'modeling/results/predictions' / SPLIT
MDL_DIR  = ROOT / 'modeling/models' / SPLIT
PRED_DIR.mkdir(parents=True, exist_ok=True)
MDL_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'AOD_550nm'
TRANSFORMS = ['raw', 'log1p']

print(f'Split: {SPLIT}')
print(f'Predictions → {PRED_DIR}')
print(f'Models      → {MDL_DIR}')

Split: holdout_time
Predictions → D:\Regional-Scale-AOD-Estimation\modeling\results\predictions\holdout_time
Models      → D:\Regional-Scale-AOD-Estimation\modeling\models\holdout_time


In [3]:
train_df = pd.read_csv(FEAT_DIR / 'train.csv')
val_df   = pd.read_csv(FEAT_DIR / 'val.csv')

feat_cols = [c for c in train_df.columns if c != TARGET]

X_train = train_df[feat_cols].values.astype(np.float32)
y_train = train_df[TARGET].values.astype(np.float64)
X_val   = val_df[feat_cols].values.astype(np.float32)
y_val   = val_df[TARGET].values.astype(np.float64)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'AOD train range: [{y_train.min():.3f}, {y_train.max():.3f}]  mean={y_train.mean():.3f}')
print(f'AOD val   range: [{y_val.min():.3f},  {y_val.max():.3f}]   mean={y_val.mean():.3f}')

Train: (4300, 3700)  |  Val: (834, 3700)
AOD train range: [0.008, 2.928]  mean=0.229
AOD val   range: [0.023,  0.989]   mean=0.194


In [4]:
# PCA-150 for linear/kernel models (fit on train only)
print('Fitting PCA-150 ...')
pca = PCA(n_components=150, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_val_pca   = pca.transform(X_val)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%')

Fitting PCA-150 ...
PCA variance explained: 99.0%


In [5]:
# Top-500 by mutual information for TabPFN (fit on train only)
print('Computing mutual information (top-500 for TabPFN) ...')
mi_scores  = mutual_info_regression(X_train, y_train, random_state=42, n_jobs=-1)
top_mi_idx = np.argsort(mi_scores)[::-1][:500]
top_mi_idx = np.sort(top_mi_idx)  # preserve column order
X_train_mi = X_train[:, top_mi_idx]
X_val_mi   = X_val[:, top_mi_idx]
print(f'Top-500 MI: X_train_mi {X_train_mi.shape}  |  X_val_mi {X_val_mi.shape}')

Computing mutual information (top-500 for TabPFN) ...
Top-500 MI: X_train_mi (4300, 500)  |  X_val_mi (834, 500)


In [6]:
# Model definitions: (name, constructor, feature_set)
# feature_set: 'raw' | 'pca' | 'mi'
# TabPFN is benchmarked separately in the TabPFN Study section below.
MODEL_DEFS = [
    # GPU boosting
    ('lgbm',    lgb.LGBMRegressor(
                    n_estimators=1000, learning_rate=0.05,
                    num_leaves=127, device='gpu', random_state=42, verbose=-1),
     'raw'),
    ('xgboost', xgb.XGBRegressor(
                    n_estimators=1000, learning_rate=0.05,
                    max_depth=6, device='cuda', random_state=42, verbosity=0),
     'raw'),
    ('catboost', cb.CatBoostRegressor(
                    iterations=1000, learning_rate=0.05,
                    depth=6, task_type='GPU', random_seed=42, verbose=0),
     'raw'),
    # CPU tree ensembles
    ('rf',      RandomForestRegressor(
                    n_estimators=500, n_jobs=-1, random_state=42),
     'raw'),
    ('extratrees', ExtraTreesRegressor(
                    n_estimators=500, n_jobs=-1, random_state=42),
     'raw'),
    ('gbm',     GradientBoostingRegressor(
                    n_estimators=500, learning_rate=0.05,
                    max_depth=5, random_state=42),
     'raw'),
    # Linear/kernel — PCA-150
    ('ridge',   Ridge(alpha=1.0),    'pca'),
    ('lasso',   Lasso(alpha=0.001),  'pca'),
    ('svr',     SVR(kernel='rbf', C=10.0, epsilon=0.01), 'pca'),
]

print(f'{len(MODEL_DEFS)} models defined  (TabPFN benchmarked separately below)')

9 models defined  (TabPFN benchmarked separately below)


In [7]:
from tqdm.auto import tqdm

results_log = []  # accumulates across both transform cells in this session

def _clear_gpu():
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

def _train_transform(transform):
    """Train all 9 models for one target transform. Appends to results_log."""
    y_tr = np.log1p(y_train) if transform == 'log1p' else y_train

    bar = tqdm(
        MODEL_DEFS, desc=f'[{transform}]', unit='model',
        bar_format='{desc} |{bar:25}| {n_fmt}/{total_fmt} {postfix}',
        leave=True,
    )
    for name, model, feat_set in bar:
        bar.set_postfix_str(f'{name} ({feat_set}) …', refresh=True)

        Xtr = X_train_pca if feat_set == 'pca' else (X_train_mi if feat_set == 'mi' else X_train)
        Xva = X_val_pca   if feat_set == 'pca' else (X_val_mi   if feat_set == 'mi' else X_val)

        try:
            m = sklearn.base.clone(model)
        except Exception:
            m = model.__class__(**model.get_params())

        t0 = time()
        try:
            m.fit(Xtr, y_tr)
            y_pred_t = m.predict(Xva)   # timing covers fit + predict
        except Exception as exc:
            bar.write(f'  ✗ [{transform}] {name:<12}  SKIPPED — {exc.__class__.__name__}: {exc}')
            results_log.append({'split': SPLIT, 'model': name, 'transform': transform,
                                'feat_set': feat_set, 'rmse': float('nan'), 'train_time_s': 0})
            del m; _clear_gpu()
            continue
        elapsed = time() - t0

        y_pred_aod = np.expm1(y_pred_t) if transform == 'log1p' else y_pred_t
        y_pred_aod = np.clip(y_pred_aod, 0.0, None)
        rmse = np.sqrt(np.mean((y_pred_aod - y_val) ** 2))

        pred_df = pd.DataFrame({
            'y_true': y_val, 'y_pred': y_pred_aod,
            'split': SPLIT, 'model': name, 'target_transform': transform,
        })
        pred_df.to_csv(PRED_DIR / f'{name}_{transform}.csv', index=False)
        joblib.dump(m, MDL_DIR / f'{name}_{transform}.joblib')

        bar.write(f'  ✓ [{transform}] {name:<12}  RMSE={rmse:.4f}  feat={feat_set}  t={elapsed:.0f}s  → saved')
        results_log.append({'split': SPLIT, 'model': name, 'transform': transform,
                            'feat_set': feat_set, 'rmse': rmse, 'train_time_s': round(elapsed, 1)})
        del m; _clear_gpu()

    bar.set_postfix_str('done ✓', refresh=True)

print('Training helper ready — run the raw and/or log1p cells below.')

Training helper ready — run the raw and/or log1p cells below.


In [8]:
# ── Train — raw ────────────────────────────────────────────────────────────────
_train_transform('raw')

[raw] |██▊                      | 1/9 , xgboost (raw) …

  ✓ [raw] lgbm          RMSE=0.0439  feat=raw  t=385s  → saved


[raw] |█████▌                   | 2/9 , catboost (raw) …

  ✓ [raw] xgboost       RMSE=0.0427  feat=raw  t=79s  → saved


[raw] |████████▎                | 3/9 , rf (raw) …      

...

  ✓ [raw] gbm           RMSE=0.0432  feat=raw  t=3664s  → saved
  ✓ [raw] ridge         RMSE=0.0529  feat=pca  t=0s  → saved


[raw] |██████████████████████▏  | 8/9 , svr (pca) …  

  ✓ [raw] lasso         RMSE=0.0515  feat=pca  t=0s  → saved


[raw] |█████████████████████████| 9/9 , svr (pca) …

  ✓ [raw] svr           RMSE=0.0376  feat=pca  t=2s  → saved


In [9]:
# ── Train — log1p ──────────────────────────────────────────────────────────────
_train_transform('log1p')

[log1p] |██▊                      | 1/9 , xgboost (raw) …

  ✓ [log1p] lgbm          RMSE=0.0461  feat=raw  t=386s  → saved


[log1p] |█████▌                   | 2/9 , catboost (raw) …

  ✓ [log1p] xgboost       RMSE=0.0441  feat=raw  t=80s  → saved


[log1p] |████████▎                | 3/9 , rf (raw) …      

...

  ✓ [log1p] gbm           RMSE=0.0442  feat=raw  t=3599s  → saved
  ✓ [log1p] ridge         RMSE=0.0455  feat=pca  t=0s  → saved


[log1p] |██████████████████████▏  | 8/9 , svr (pca) …  

  ✓ [log1p] lasso         RMSE=0.0473  feat=pca  t=0s  → saved


[log1p] |█████████████████████████| 9/9 , svr (pca) …

  ✓ [log1p] svr           RMSE=0.0366  feat=pca  t=2s  → saved


In [10]:
rows = []
for p in sorted(PRED_DIR.glob('*.csv')):
    df = pd.read_csv(p)
    rmse = np.sqrt(np.mean((df['y_pred'] - df['y_true'])**2))
    rows.append({'model': df['model'].iloc[0], 'transform': df['target_transform'].iloc[0], 'rmse': round(rmse, 4)})

print(f'\nSummary — split={SPLIT}')
print(pd.DataFrame(rows).sort_values(['transform', 'rmse']).to_string(index=False))


Summary — split=holdout_time
     model transform   rmse
       svr     log1p 0.0366
  catboost     log1p 0.0420
   xgboost     log1p 0.0441
       gbm     log1p 0.0442
     ridge     log1p 0.0455
      lgbm     log1p 0.0461
extratrees     log1p 0.0463
     lasso     log1p 0.0473
        rf     log1p 0.0514
       svr       raw 0.0376
   xgboost       raw 0.0427
       gbm       raw 0.0432
      lgbm       raw 0.0439
extratrees       raw 0.0454
  catboost       raw 0.0456
        rf       raw 0.0485
     lasso       raw 0.0515
     ridge       raw 0.0529


---
## TabPFN Study

TabPFN is an in-context learner: `fit()` only stores training data (~instant), while `predict()` runs a full transformer forward pass with all N training samples as context for every test point. The test set is batched in chunks of 100 to avoid GPU OOM.

This section tests 4 feature configurations × 2 transforms (8 runs total):

| Config | Features |
|--------|----------|
| `pca150` | PCA-150 components (99% variance) |
| `mi100` – `mi300` | Top-N features by mutual information |

Each run reports `fit_s` and `pred_s` separately. Predictions are saved as `tabpfn_{feat}_{transform}.csv` and automatically picked up by `02_evaluate.ipynb`.

In [11]:
# ── TabPFN Study — Run ─────────────────────────────────────────────────────────
# Tests 4 feature configs × 2 transforms = 8 runs.
# fit and predict are timed separately — TabPFN's fit() just stores data;
# predict() is where all transformer compute happens.
#
# Uses v2.6 (latest). Test set is batched in chunks of 100 to avoid GPU OOM.
#
# Predictions saved as  tabpfn_{feat}_{transform}.csv  (model name = tabpfn_{feat})
# so 02_evaluate.ipynb picks them all up and compares them side by side.

from tabpfn.constants import ModelVersion

TABPFN_PRED_BATCH = 100   # test-set chunk size to avoid GPU OOM

def _tabpfn_predict_batched(m, Xva, batch_size):
    parts = []
    for i in range(0, len(Xva), batch_size):
        parts.append(m.predict(Xva[i:i + batch_size]))
    return np.concatenate(parts)

def _mi_slice(n):
    idx = np.sort(np.argsort(mi_scores)[::-1][:n])
    return X_train[:, idx], X_val[:, idx]

feat_configs = [
    ('pca150', X_train_pca,   X_val_pca),
    ('mi100',  *_mi_slice(100)),
    ('mi200',  *_mi_slice(200)),
    ('mi300',  *_mi_slice(300)),
]

tabpfn_rows = []
runs = [(tag, Xtr, Xva, tf)
        for tag, Xtr, Xva in feat_configs
        for tf in ['raw', 'log1p']]

bar = tqdm(runs, desc='TabPFN', unit='run',
           bar_format='{desc} |{bar:25}| {n_fmt}/{total_fmt} {postfix}', leave=True)

for tag, Xtr, Xva, transform in bar:
    bar.set_postfix_str(f'{tag}/{transform} …', refresh=True)
    y_tr = np.log1p(y_train) if transform == 'log1p' else y_train

    try:
        m = TabPFNRegressor.create_default_for_version(
            ModelVersion.V2_6,
            device='cuda',
            random_state=42,
        )

        t0 = time()
        m.fit(Xtr, y_tr)
        fit_s = time() - t0

        t0 = time()
        y_pred_t = _tabpfn_predict_batched(m, Xva, TABPFN_PRED_BATCH)
        pred_s = time() - t0

    except Exception as exc:
        bar.write(f'  ✗ tabpfn {tag}/{transform}  SKIPPED — {exc.__class__.__name__}: {exc}')
        tabpfn_rows.append({'feat': tag, 'n_feat': Xtr.shape[1], 'transform': transform,
                             'rmse': float('nan'), 'fit_s': 0, 'pred_s': 0})
        _clear_gpu()
        continue

    y_pred_aod = np.expm1(y_pred_t) if transform == 'log1p' else y_pred_t
    y_pred_aod = np.clip(y_pred_aod, 0.0, None)
    rmse = np.sqrt(np.mean((y_pred_aod - y_val) ** 2))

    pred_df = pd.DataFrame({
        'y_true': y_val, 'y_pred': y_pred_aod,
        'split': SPLIT, 'model': f'tabpfn_{tag}', 'target_transform': transform,
    })
    pred_df.to_csv(PRED_DIR / f'tabpfn_{tag}_{transform}.csv', index=False)
    joblib.dump(m, MDL_DIR / f'tabpfn_{tag}_{transform}.joblib')

    bar.write(f'  ✓ tabpfn {tag:8s}/{transform:5s}  '
              f'n={Xtr.shape[1]:3d}  RMSE={rmse:.4f}  '
              f'fit={fit_s:.1f}s  pred={pred_s:.1f}s  total={fit_s+pred_s:.1f}s')
    tabpfn_rows.append({'feat': tag, 'n_feat': Xtr.shape[1], 'transform': transform,
                         'rmse': rmse, 'fit_s': round(fit_s, 1), 'pred_s': round(pred_s, 1)})
    del m; _clear_gpu()

bar.set_postfix_str('done ✓', refresh=True)

TabPFN |███▏                     | 1/8 , pca150/log1p …

  ✓ tabpfn pca150  /raw    n=150  RMSE=0.0367  fit=3.6s  pred=359.1s  total=362.8s


TabPFN |██████▎                  | 2/8 , mi100/raw …   

  ✓ tabpfn pca150  /log1p  n=150  RMSE=0.0363  fit=3.3s  pred=358.3s  total=361.6s


TabPFN |█████████▍               | 3/8 , mi100/log1p …

...

  ✓ tabpfn mi200   /log1p  n=200  RMSE=0.0348  fit=4.1s  pred=1834.4s  total=1838.6s


TabPFN |██████████████████▊      | 6/8 , mi300/raw …  error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolu

  ✓ tabpfn mi300   /raw    n=300  RMSE=0.0345  fit=4.7s  pred=1381.4s  total=1386.1s


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', po

  ✓ tabpfn mi300   /log1p  n=300  RMSE=0.0345  fit=4.6s  pred=460.3s  total=464.9s


In [12]:
# ── TabPFN Study — Results ─────────────────────────────────────────────────────
tab_df = pd.DataFrame(tabpfn_rows)
tab_df['total_s'] = tab_df['fit_s'] + tab_df['pred_s']

print('=== TabPFN Study — fit vs predict timing breakdown ===')
print(tab_df[['feat', 'n_feat', 'transform', 'rmse',
              'fit_s', 'pred_s', 'total_s']].to_string(index=False))

# Best config per transform
for tf in ['raw', 'log1p']:
    sub = tab_df[tab_df['transform'] == tf].dropna(subset=['rmse'])
    if len(sub):
        b = sub.sort_values('rmse').iloc[0]
        print(f'\nBest [{tf}]:  feat={b.feat}  n={int(b.n_feat)}  '
              f'RMSE={b.rmse:.4f}  fit={b.fit_s:.0f}s  pred={b.pred_s:.0f}s')

print(f'\nAll TabPFN predictions saved to: {PRED_DIR}')
print('02_evaluate.ipynb will pick up tabpfn_* files as separate model entries for comparison.')

=== TabPFN Study — fit vs predict timing breakdown ===
  feat  n_feat transform     rmse  fit_s  pred_s  total_s
pca150     150       raw 0.036653    3.6   359.1    362.7
pca150     150     log1p 0.036321    3.3   358.3    361.6
 mi100     100       raw 0.037335    2.0   243.8    245.8
 mi100     100     log1p 0.037181    2.0   243.7    245.7
 mi200     200       raw 0.034801    4.0  1073.1   1077.1
 mi200     200     log1p 0.034766    4.1  1834.4   1838.5
 mi300     300       raw 0.034469    4.7  1381.4   1386.1
 mi300     300     log1p 0.034525    4.6   460.3    464.9

Best [raw]:  feat=mi300  n=300  RMSE=0.0345  fit=5s  pred=1381s

Best [log1p]:  feat=mi300  n=300  RMSE=0.0345  fit=5s  pred=460s

All TabPFN predictions saved to: D:\Regional-Scale-AOD-Estimation\modeling\results\predictions\holdout_time
02_evaluate.ipynb will pick up tabpfn_* files as separate model entries for comparison.


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
